# ⚙️ Step 0: Configuration

Configure the training pipeline below. **Default: DATA_SOURCE = "upload"** — trains on your real database exports.

Set `DATA_SOURCE = "synthetic"` ONLY if you want to fall back to the built-in synthetic generator (for testing/debugging only).

⚠️ **IMPORTANT**: If your database contains seeded/demo rows flagged `is_synthetic = 1`, they are included by default (`INCLUDE_SYNTHETIC = True`). This means the model will learn the seeder's patterns, not purely real behavior. Set `INCLUDE_SYNTHETIC = False` to exclude them — but only if you have enough real rows (see warning below).

In [ ]:
# ============================
# CONFIGURATION — EDIT THESE VALUES
# ============================

DATA_SOURCE = "upload"  # "upload" (real CSVs) or "synthetic" (built-in generator, fallback only)
INCLUDE_SYNTHETIC = True  # Include is_synthetic=1 rows from your database export
LABEL_HORIZON_DAYS = 180   # Reoffence window in days
CUTOFF_EVERY_DAYS = 14     # Spacing between time-based training cutoffs
MIN_TRAIN_ROWS = 2000      # Minimum training rows required

# Fine per demerit point (KES) — must match your backend
FINE_PER_POINT = 500

# Sanction thresholds (applied to final risk score)
RISK_WARNING_THRESHOLD = 0.40
RISK_TRAINING_THRESHOLD = 0.55
RISK_SANCTION_THRESHOLD = 0.70

# Points-floor anchors for risk floor calculation
POINTS_FLOOR_ANCHORS = {
    "warning": {"points": 10, "score": 0.40},
    "restriction": {"points": 20, "score": 0.55},
    "suspension": {"points": 30, "score": 0.70},
}

# Offense → points mapping (must match your backend offense catalog)
OFFENSE_POINTS = {
    "speeding": 4,
    "illegal_parking": 2,
    "dui": 10,
    "reckless_driving": 8,
    "red_light": 5,
    "no_seatbelt": 2,
    "phone_use": 3,
    "driving_without_license": 8,
    "overloading": 4,
    "no_insurance": 6,
}

# Random seed for reproducibility
RANDOM_STATE = 42

print("✓ Configuration loaded")
print(f"  DATA_SOURCE = {DATA_SOURCE}")
print(f"  INCLUDE_SYNTHETIC = {INCLUDE_SYNTHETIC}")
print(f"  LABEL_HORIZON_DAYS = {LABEL_HORIZON_DAYS}")
print(f"  CUTOFF_EVERY_DAYS = {CUTOFF_EVERY_DAYS}")
print(f"  MIN_TRAIN_ROWS = {MIN_TRAIN_ROWS}")
print(f"  FINE_PER_POINT = {FINE_PER_POINT} KES")
print(f"  RISK thresholds: warning={RISK_WARNING_THRESHOLD}, training={RISK_TRAINING_THRESHOLD}, suspension={RISK_SANCTION_THRESHOLD}")
print(f"  Points-floor anchors: {POINTS_FLOOR_ANCHORS}")

# 📂 Step 1: Load Your Database Exports

Export the following three tables from your MySQL database (`traffic_demerit`) via **phpMyAdmin → Export → CSV** or the `mysql` CLI:

```sql
-- 1. Violations
SELECT id AS violation_id, driver_id, offense_type, speed_recorded, posted_speed_limit,
       zone_type, weather_conditions, points_assigned, occurred_at, is_synthetic
FROM violations WHERE deleted_at IS NULL;

-- 2. Payments
SELECT id AS payment_id, violation_id, driver_id, amount, amount_paid, status, paid_at, is_synthetic
FROM payments WHERE deleted_at IS NULL;

-- 3. Driver profiles
SELECT id AS driver_id, license_issue_date, is_synthetic
FROM driver_profiles WHERE deleted_at IS NULL;
```

**Save as:** `violations.csv`, `payments.csv`, `drivers.csv`

If your files have different names, the cell below will try to detect them by keyword matching.

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import json
from pathlib import Path

print("📤 Please upload your three CSV export files:")
print("   1. violations.csv")
print("   2. payments.csv")
print("   3. drivers.csv")
print("   (Filenames can vary — they will be auto-detected by keyword)")
print()

uploaded = files.upload()

# Detect files by keyword in filename
uploaded_files = list(uploaded.keys())
print(f"\n💾 Uploaded {len(uploaded_files)} file(s): {uploaded_files}")

file_map = {}
for fname in uploaded_files:
    lower = fname.lower()
    if 'viol' in lower:
        file_map['violations'] = fname
    elif 'pay' in lower:
        file_map['payments'] = fname
    elif 'driver' in lower:
        file_map['drivers'] = fname
    else:
        print(f"⚠️  Unrecognized file: {fname}")

print(f"\n🗭️  Detected file mapping:")
for k, v in file_map.items():
    print(f"   {k}: {v}")

# Check we have all three required files
required = ['violations', 'payments', 'drivers']
missing = [k for k in required if k not in file_map]
if missing:
    raise FileNotFoundError(
        f"❌ Missing required file(s): {missing}.\n"
        f"   Please export and upload: violations.csv, payments.csv, drivers.csv\n"
        f"   (Filenames can vary as long as they contain the keywords.)"
    )

# Read CSVs
violations_df = pd.read_csv(file_map['violations'])
payments_df = pd.read_csv(file_map['payments'])
drivers_df = pd.read_csv(file_map['drivers'])

print(f"\n📅 Loaded:")
print(f"   violations: {len(violations_df)} rows")
print(f"   payments: {len(payments_df)} rows")
print(f"   drivers: {len(drivers_df)} rows")

# 📁 Step 1b: Column Mapping & Validation

Map your export column names to the internal schema expected by the pipeline.
Common aliases are handled automatically (e.g., `offence_type` → `offense_type`, `weather` → `weather_conditions`).

**Required columns after mapping:**
- `violations`: violation_id, driver_id, offense_type, speed_recorded, posted_speed_limit, zone_type, weather_conditions, points_assigned, occurred_at, is_synthetic
- `payments`: payment_id, violation_id, driver_id, amount, amount_paid, status, paid_at, is_synthetic
- `drivers`: driver_id, license_issue_date, is_synthetic

In [ ]:
# Column alias mapping (your export name → internal name)
COLUMN_ALIASES = {
    # Violations
    'offence_type': 'offense_type',
    'offense_type': 'offense_type',
    'speed_limit': 'posted_speed_limit',
    'weather': 'weather_conditions',
    'weather_condition': 'weather_conditions',
    'violation_date_time': 'occurred_at',
    'violation_date': 'occurred_at',
    'occurred_date': 'occurred_at',
    # Payments
    'payment_id': 'payment_id',
    'violation_id': 'violation_id',
    'driver_id': 'driver_id',
    'amount': 'amount',
    'amount_paid': 'amount_paid',
    'status': 'status',
    'paid_at': 'paid_at',
    'is_synthetic': 'is_synthetic',
    # Drivers
    'driver_id': 'driver_id',
    'license_issue_date': 'license_issue_date',
    'license_issue': 'license_issue_date',
    'is_synthetic': 'is_synthetic',
}

def apply_column_mapping(df, entity_name):
    """Rename columns using aliases, validate required columns exist."""
    df = df.copy()
    
    # Build reverse mapping: any column name that appears in aliases gets renamed
    rename_map = {}
    for col in df.columns:
        col_clean = col.strip().lower()
        if col_clean in COLUMN_ALIASES:
            rename_map[col] = COLUMN_ALIASES[col_clean]
        elif col_clean in [v.lower() for v in COLUMN_ALIASES.values()]:
            # Already an internal name, keep as-is
            pass
    
    df = df.rename(columns=rename_map)
    
    # Define required columns per entity
    required_cols = {
        'violations': ['violation_id', 'driver_id', 'offense_type', 'speed_recorded', 
                      'posted_speed_limit', 'zone_type', 'weather_conditions', 
                      'points_assigned', 'occurred_at', 'is_synthetic'],
        'payments': ['payment_id', 'violation_id', 'driver_id', 'amount', 
                    'amount_paid', 'status', 'paid_at', 'is_synthetic'],
        'drivers': ['driver_id', 'license_issue_date', 'is_synthetic']
    }
    
    missing = [c for c in required_cols[entity_name] if c not in df.columns]
    if missing:
        raise ValueError(
            f"❌ {entity_name} is missing required columns: {missing}\n"
            f"   Available columns: {list(df.columns)}\n"
            f"   Please check your export and column naming."
        )
    
    return df

# Apply mapping to all three DataFrames
violations_df = apply_column_mapping(violations_df, 'violations')
payments_df = apply_column_mapping(payments_df, 'payments')
drivers_df = apply_column_mapping(drivers_df, 'drivers')

print("✓ Column mapping applied successfully")
print(f"\nViolations columns: {list(violations_df.columns)}")
print(f"Payments columns: {list(payments_df.columns)}")
print(f"Drivers columns: {list(drivers_df.columns)}")

# 📊 Step 2: Data Readiness & EDA

Parse dates, filter synthetic rows if configured, and print a data-readiness report.

If `INCLUDE_SYNTHETIC = False`, rows flagged `is_synthetic = 1` are excluded.
If that leaves fewer than 200 violations, the notebook stops with a clear message.

In [ ]:
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go

print("📋 Parsing dates and validating data...")
print("=" * 60)

# Parse dates safely
violations_df['occurred_at'] = pd.to_datetime(violations_df['occurred_at'], errors='coerce')
payments_df['paid_at'] = pd.to_datetime(payments_df['paid_at'], errors='coerce')
drivers_df['license_issue_date'] = pd.to_datetime(drivers_df['license_issue_date'], errors='coerce')

# Count unparseable dates
bad_violation_dates = violations_df['occurred_at'].isna().sum()
bad_payment_dates = payments_df['paid_at'].isna().sum()
bad_driver_dates = drivers_df['license_issue_date'].isna().sum()

if bad_violation_dates > 0:
    print(f"⚠️  {bad_violation_dates} violation rows had unparseable occurred_at and will be dropped.")
    violations_df = violations_df.dropna(subset=['occurred_at'])

if bad_payment_dates > 0:
    print(f"⚠️  {bad_payment_dates} payment rows had unparseable paid_at.")

if bad_driver_dates > 0:
    print(f"⚠️  {bad_driver_dates} driver rows had unparseable license_issue_date.")

# Filter synthetic rows if configured
if not INCLUDE_SYNTHETIC:
    before = len(violations_df)
    violations_df = violations_df[violations_df['is_synthetic'] == 0]
    after = len(violations_df)
    print(f"\n📭 Filtered synthetic rows: {before} → {after} ({before - after} removed)")
    
    if after < 200:
        print(f"\n❌ ERROR: Only {after} real (non-synthetic) violations remain.")
        print("   This is too few rows for meaningful training.")
        print("   Set INCLUDE_SYNTHETIC = True to include seeded/demo rows, or export more data.")
        raise SystemExit("Insufficient real data for training.")

# Compute date range
first_date = violations_df['occurred_at'].min()
last_date = violations_df['occurred_at'].max()

# Compute basic stats
total_drivers = drivers_df['driver_id'].nunique()
total_violations = len(violations_df)
total_payments = len(payments_df)
drivers_with_violations = violations_df['driver_id'].nunique()
drivers_without_violations = total_drivers - drivers_with_violations
violations_per_driver = violations_df.groupby('driver_id').size()

# Label distribution
print("\n📈 DATA READINESS REPORT")
print("=" * 60)
print(f"Date range: {first_date.date()} to {last_date.date()}")
print(f"Total drivers: {total_drivers}")
print(f"Drivers with violations: {drivers_with_violations}")
print(f"Drivers without violations: {drivers_without_violations}")
print(f"Total violations: {total_violations}")
print(f"Total payments: {total_payments}")
print(f"\nViolations per driver:")
print(f"   Min: {violations_per_driver.min()}")
print(f"   Median: {violations_per_driver.median():.1f}")
print(f"   Max: {violations_per_driver.max()}")
print(f"   Mean: {violations_per_driver.mean():.1f}")

# Drivers with 2+ violations (can be labeled)
drivers_multi = (violations_per_driver >= 2).sum()
print(f"\nDrivers with 2+ violations: {drivers_multi}")
if drivers_multi < 100:
    print(f"   ⚠️  WARNING: Fewer than 100 drivers have 2+ violations. "
          "The model will be weak for recidivism prediction.")

# Bar chart: violations per month
violations_df['month'] = violations_df['occurred_at'].dt.to_period('M')
monthly_counts = violations_df.groupby('month').size().reset_index(name='count')
monthly_counts['month'] = monthly_counts['month'].astype(str)

fig = px.bar(monthly_counts, x='month', y='count',
             title='Violations per Month',
             labels={'month': 'Month', 'count': 'Number of Violations'},
             color_discrete_sequence=['#10b981'])
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print("\n✓ Data readiness check complete.")

# 💻 Step 2: Feature Engineering

Build per-driver features at each violation's `occurred_at` (point-in-time, no leakage):
- violation_count_12m, violation_count_90d
- weighted_points_12m (severity-weighted)
- cumulative_active_demerit_points (ledger balance at that time)
- severe_offence_share, avg_speed_over_limit, max_speed_over_limit
- days_since_last_offense, mean_gap_between_offences, trend (last 90d vs prior 90d)
- share_night, share_school_zone, share_bad_weather
- unpaid_fine_ratio, avg_payment_delay_days, appeals_lost
- new_driver flag (< 2 violations)

Labels: `reoffended_within_180_days` = 1 if the driver has another violation within 180 days after this violation's `occurred_at`.

In [ ]:
print("💻 Building per-driver features at each violation's occurred_at...")
print("=" * 60)

# Sort violations by occurred_at for consistent processing
violations_df = violations_df.sort_values('occurred_at').reset_index(drop=True)

# Pre-compute driver-level aggregations for efficiency
driver_violations = violations_df.groupby('driver_id')

# Pre-compute payment stats per driver-violation
payments_df['paid_at'] = pd.to_datetime(payments_df['paid_at'], errors='coerce')
payments_with_amount = payments_df[payments_df['status'] == 'completed'].copy()

# Build feature rows
feature_rows = []

for idx, v in violations_df.iterrows():
    driver_id = v['driver_id']
    occurred_at = v['occurred_at']
    
    # Driver's violations up to and including this one
    driver_v = driver_violations.get_group(driver_id)
    driver_v_past = driver_v[driver_v['occurred_at'] <= occurred_at]
    driver_v_prior_90d = driver_v_past[driver_v_past['occurred_at'] >= occurred_at - timedelta(days=90)]
    driver_v_prior_180d = driver_v_past[driver_v_past['occurred_at'] >= occurred_at - timedelta(days=180)]
    driver_v_prior_180d_before = driver_v_past[(driver_v_past['occurred_at'] >= occurred_at - timedelta(days=180)) & 
                                               (driver_v_past['occurred_at'] < occurred_at - timedelta(days=90))]
    
    violation_count_12m = len(driver_v_past[driver_v_past['occurred_at'] >= occurred_at - timedelta(days=365)])
    violation_count_90d = len(driver_v_prior_90d)
    
    # Severity-weighted points (last 12 months)
    weighted_points_12m = 0.0
    severe_count = 0
    severe_offence_types = {'dui', 'reckless_driving', 'red_light', 'driving_without_license'}
    
    for _, dv in driver_v_past[driver_v_past['occurred_at'] >= occurred_at - timedelta(days=365)].iterrows():
        offense = dv['offense_type']
        weight = OFFENSE_POINTS.get(offense, 1)
        weighted_points_12m += weight * float(dv['points_assigned'])
        if offense in severe_offence_types:
            severe_count += 1
    
    severe_offence_share = severe_count / violation_count_12m if violation_count_12m > 0 else 0.0
    
    # Cumulative active demerit points (ledger balance at this violation)
    cumulative_points = driver_v_past['points_assigned'].sum()
    
    # Speed stats (speeding violations only)
    speeding = driver_v_past[(driver_v_past['offense_type'] == 'speeding') & 
                            (driver_v_past['speed_recorded'].notna()) & 
                            (driver_v_past['posted_speed_limit'].notna())]
    avg_speed_over_limit = 0.0
    max_speed_over_limit = 0.0
    if len(speeding) > 0:
        excesses = (speeding['speed_recorded'].astype(float) - speeding['posted_speed_limit'].astype(float)).clip(lower=0)
        avg_speed_over_limit = excesses.mean()
        max_speed_over_limit = excesses.max()
    
    # Days since last offense
    if len(driver_v_past) > 1:
        days_since_last = (occurred_at - driver_v_past.iloc[-2]['occurred_at']).days
    else:
        days_since_last = 365  # Default for first offense
    
    # Mean gap between offences
    if len(driver_v_past) >= 2:
        sorted_dv = driver_v_past.sort_values('occurred_at')
        gaps = []
        for i in range(1, len(sorted_dv)):
            gaps.append((sorted_dv.iloc[i]['occurred_at'] - sorted_dv.iloc[i-1]['occurred_at']).days)
        mean_gap = np.mean(gaps) if gaps else 0.0
    else:
        mean_gap = 0.0
    
    # Trend: last 90d vs prior 90d
    if len(driver_v_prior_180d_before) > 0:
        trend = (violation_count_90d - len(driver_v_prior_180d_before)) / len(driver_v_prior_180d_before)
    elif violation_count_90d > 0:
        trend = 1.0
    else:
        trend = 0.0
    
    # Time/zone/weather shares
    share_night = (driver_v_past['weather_conditions'] == 'night').mean() if len(driver_v_past) > 0 else 0.0
    share_school_zone = (driver_v_past['zone_type'] == 'school_zone').mean() if len(driver_v_past) > 0 else 0.0
    share_bad_weather = driver_v_past['weather_conditions'].isin(['rain', 'fog', 'storm']).mean() if len(driver_v_past) > 0 else 0.0
    
    # Payment behavior for this driver up to this violation
    driver_payments = payments_df[payments_df['driver_id'] == driver_id]
    driver_paid = driver_payments[driver_payments['status'] == 'completed']
    
    # Total fines for this driver's violations up to now
    total_fines = driver_v_past['points_assigned'].sum() * FINE_PER_POINT
    paid_fines = 0.0
    payment_delays = []
    
    for _, dv in driver_v_past.iterrows():
        dv_payments = driver_paid[driver_paid['violation_id'] == dv['violation_id']]
        fine = dv['points_assigned'] * FINE_PER_POINT
        paid_fines += fine
        for _, p in dv_payments.iterrows():
            if p['paid_at'] and dv['occurred_at']:
                delay = (p['paid_at'] - dv['occurred_at']).days
                payment_delays.append(delay)
    
    unpaid_fine_ratio = max(0.0, min(1.0, 1.0 - (paid_fines / total_fines))) if total_fines > 0 else 0.0
    avg_payment_delay = np.mean(payment_delays) if payment_delays else 0.0
    
    # Appeals (if appeals data exists, otherwise 0)
    appeals_lost = 0  # No appeals data in export
    
    # New driver flag
    new_driver = violation_count_12m < 2
    
    # Label: reoffended within 180 days AFTER this violation
    driver_future = driver_v[driver_v['occurred_at'] > occurred_at]
    driver_future_within_180 = driver_future[driver_future['occurred_at'] <= occurred_at + timedelta(days=LABEL_HORIZON_DAYS)]
    reoffended = 1 if len(driver_future_within_180) > 0 else 0
    
    feature_rows.append({
        'driver_id': driver_id,
        'violation_id': v['violation_id'],
        'occurred_at': occurred_at,
        'days_since_last_offense': days_since_last,
        'speed_recorded': float(v['speed_recorded']) if pd.notna(v['speed_recorded']) else 0.0,
        'posted_speed_limit': float(v['posted_speed_limit']) if pd.notna(v['posted_speed_limit']) else 80.0,
        'zone_type': v['zone_type'],
        'weather_conditions': v['weather_conditions'],
        'cumulative_active_demerit_points': int(cumulative_points),
        'past_fine_payment_delays_days': round(float(avg_payment_delay), 2),
        'violation_count_12m': violation_count_12m,
        'violation_count_90d': violation_count_90d,
        'weighted_points_12m': round(weighted_points_12m, 2),
        'severe_offence_share': round(severe_offence_share, 4),
        'avg_speed_over_limit': round(avg_speed_over_limit, 2),
        'max_speed_over_limit': round(max_speed_over_limit, 2),
        'mean_gap_between_offences': round(mean_gap, 2),
        'trend': round(trend, 4),
        'share_night': round(share_night, 4),
        'share_school_zone': round(share_school_zone, 4),
        'share_bad_weather': round(share_bad_weather, 4),
        'unpaid_fine_ratio': round(unpaid_fine_ratio, 4),
        'avg_payment_delay_days': round(avg_payment_delay, 2),
        'appeals_lost': appeals_lost,
        'new_driver': int(new_driver),
        'reoffended_within_180_days': reoffended,
        'is_synthetic': int(v['is_synthetic']) if 'is_synthetic' in v and pd.notna(v['is_synthetic']) else 0,
    })

features_df = pd.DataFrame(feature_rows)
print(f"\n✓ Built {len(features_df)} feature rows (one per violation)")
print(f"   Features per row: {len(features_df.columns)}")

# Show label distribution
label_dist = features_df['reoffended_within_180_days'].value_counts()
print(f"\nLabel distribution (reoffended_within_{LABEL_HORIZON_DAYS}d):")
for k, v in label_dist.items():
    print(f"   Class {k}: {v} ({v/len(features_df)*100:.1f}%)")

if label_dist.get(1, 0) < 100:
    print(f"\n⚠️  WARNING: Only {label_dist.get(1, 0)} positive (reoffended) labels.")
    print("   Model performance may be unreliable with so few positives.")

print("\n✓ Feature engineering complete.")

# 📅 Step 3: Time-Based Train/Test Split

No random split — we simulate production: train on earlier cutoffs, test on later cutoffs.

SPLIT_DATE is computed from your data:
- First usable cutoff = first_date + 90 days
- Last valid cutoff = last_date - 180 days (LABEL_HORIZON_DAYS)
- SPLIT_DATE = 75% of the way between them

All cutoffs ≤ SPLIT_DATE - 180 days → training
All cutoffs ≥ SPLIT_DATE → testing

In [ ]:
print("📅 Computing time-based split...")
print("=" * 60)

first_date = violations_df['occurred_at'].min()
last_date = violations_df['occurred_at'].max()

# First usable cutoff: need at least 90 days of history before first prediction
first_usable_cutoff = first_date + timedelta(days=90)

# Last valid cutoff: need 180 days after cutoff to label
last_valid_cutoff = last_date - timedelta(days=LABEL_HORIZON_DAYS)

if last_valid_cutoff <= first_usable_cutoff:
    print(f"❌ ERROR: Not enough data span.")
    print(f"   First date: {first_date.date()}")
    print(f"   Last date: {last_date.date()}")
    print(f"   First usable cutoff: {first_usable_cutoff.date()}")
    print(f"   Last valid cutoff: {last_valid_cutoff.date()}")
    print("   Need at least 90 + 180 = 270 days of history.")
    raise SystemExit("Insufficient date range for time-based split.")

# SPLIT_DATE = 75% of the way between first_usable and last_valid
total_span = (last_valid_cutoff - first_usable_cutoff).days
split_offset = int(0.75 * total_span)
SPLIT_DATE = first_usable_cutoff + timedelta(days=split_offset)

print(f"First date in data: {first_date.date()}")
print(f"Last date in data: {last_date.date()}")
print(f"First usable cutoff (first_date + 90d): {first_usable_cutoff.date()}")
print(f"Last valid cutoff (last_date - {LABEL_HORIZON_DAYS}d): {last_valid_cutoff.date()}")
print(f"SPLIT_DATE (75% of span): {SPLIT_DATE.date()}")

# Generate cutoff dates every CUTOFF_EVERY_DAYS days
cutoffs = []
current = first_usable_cutoff
while current <= last_valid_cutoff:
    cutoffs.append(current)
    current += timedelta(days=CUTOFF_EVERY_DAYS)

print(f"\nCutoff spacing: {CUTOFF_EVERY_DAYS} days")
print(f"Total cutoffs: {len(cutoffs)}")

# If too few cutoffs, halve spacing
if len(cutoffs) < 5:
    CUTOFF_EVERY_DAYS = max(7, CUTOFF_EVERY_DAYS // 2)
    cutoffs = []
    current = first_usable_cutoff
    while current <= last_valid_cutoff:
        cutoffs.append(current)
        current += timedelta(days=CUTOFF_EVERY_DAYS)
    print(f"\n⚠️  Halved spacing to {CUTOFF_EVERY_DAYS} days → {len(cutoffs)} cutoffs")
    if len(cutoffs) < 5:
        print("   ⚠️  Still very few cutoffs — results may be unstable.")

# Split train/test
train_cutoffs = [c for c in cutoffs if c <= SPLIT_DATE - timedelta(days=LABEL_HORIZON_DAYS)]
test_cutoffs = [c for c in cutoffs if c >= SPLIT_DATE]

print(f"\nTrain cutoffs (≤ {SPLIT_DATE.date() - timedelta(days=LABEL_HORIZON_DAYS).date()}): {len(train_cutoffs)}")
print(f"Test cutoffs (≥ {SPLIT_DATE.date()}): {len(test_cutoffs)}")

# Build training and test sets
train_rows = []
test_rows = []

for _, row in features_df.iterrows():
    occurred_at = row['occurred_at']
    
    # Assign based on occurred_at date
    if occurred_at <= SPLIT_DATE - timedelta(days=LABEL_HORIZON_DAYS):
        train_rows.append(row)
    elif occurred_at >= SPLIT_DATE:
        test_rows.append(row)
    # Violations between SPLIT_DATE - 180d and SPLIT_DATE are excluded
    # (their label window overlaps the split boundary)

train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)

print(f"\nTraining rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")

if len(train_df) < MIN_TRAIN_ROWS:
    print(f"\n⚠️  WARNING: Only {len(train_df)} training rows (minimum {MIN_TRAIN_ROWS} recommended).")
    print("   Results may be unreliable.")

# Label distribution in train/test
print(f"\nTrain label distribution:")
print(train_df['reoffended_within_180_days'].value_counts())
print(f"\nTest label distribution:")
print(test_df['reoffended_within_180_days'].value_counts())

# Check class balance in train
train_pos_rate = train_df['reoffended_within_180_days'].mean() if len(train_df) > 0 else 0
test_pos_rate = test_df['reoffended_within_180_days'].mean() if len(test_df) > 0 else 0

if train_pos_rate < 0.05 or train_pos_rate > 0.95:
    print(f"\n⚠️  WARNING: Train positive rate ({train_pos_rate:.1%}) is outside 5%-95% range.")
    print("   Model may be biased toward the majority class.")

print("\n✓ Train/test split complete.")

# 🢻 Step 4: Supervised Risk Model Training

Candidate models:
- Logistic Regression
- Random Forest
- Gradient Boosting

Ablation study:
- (A) Original 6 features
- (B) A + segmentation features
- (C) A + rule-hit features
- (D) A + both

Probabilities are calibrated with CalibratedClassifierCV so the 0.40/0.55/0.70 thresholds are meaningful.

Explainability: SHAP values (or permutation importance fallback).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, brier_score_loss,
    classification_report, confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance
import shap

print("🢻 Building supervised risk model...")
print("=" * 60)

# Define feature sets
ORIGINAL_6 = [
    'days_since_last_offense',
    'speed_recorded',
    'zone_type',
    'weather_conditions',
    'cumulative_active_demerit_points',
    'past_fine_payment_delays_days',
]

SEGMENT_FEATURES = [
    'violation_count_12m',
    'violation_count_90d',
    'weighted_points_12m',
    'severe_offence_share',
    'avg_speed_over_limit',
    'max_speed_over_limit',
    'mean_gap_between_offences',
    'trend',
    'share_night',
    'share_school_zone',
    'share_bad_weather',
    'unpaid_fine_ratio',
    'avg_payment_delay_days',
    'new_driver',
]

RULE_FEATURE = 'appeals_lost'  # Proxy for rule hits (appeals filed)

feature_sets = {
    'A_original_6': ORIGINAL_6,
    'B_plus_segmentation': ORIGINAL_6 + SEGMENT_FEATURES,
    'C_plus_rules': ORIGINAL_6 + [RULE_FEATURE],
    'D_plus_both': ORIGINAL_6 + SEGMENT_FEATURES + [RULE_FEATURE],
}

RANDOM_STATE = 42
N_SPLITS = min(5, max(2, len(train_df) // 40))  # At least 2 folds

def build_pipeline(feature_cols, numeric_cols, categorical_cols, model_name):
    preprocess = ColumnTransformer(
        transformers=[
            (
                'num',
                Pipeline([('imputer', SimpleImputer(strategy='median'))]),
                numeric_cols,
            ),
            (
                'cat',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore')),
                ]),
                categorical_cols,
            ),
        ]
    )
    
    if model_name == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=300, max_depth=10, min_samples_leaf=3,
            random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'
        )
    elif model_name == 'gradient_boosting':
        model = GradientBoostingClassifier(
            n_estimators=200, max_depth=5, random_state=RANDOM_STATE
        )
    elif model_name == 'logistic_regression':
        model = LogisticRegression(
            max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced'
        )
    else:
        raise ValueError(f'Unknown model: {model_name}')
    
    return Pipeline([('preprocess', preprocess), ('clf', model)])

# Ablation study
ablation_results = []

for fs_name, fs_cols in feature_sets.items():
    print(f"\n--- Feature Set: {fs_name} ---")
    
    avail_cols = [c for c in fs_cols if c in train_df.columns]
    missing = [c for c in fs_cols if c not in train_df.columns]
    if missing:
        print(f"   Missing columns: {missing}")
    
    X_train = train_df[avail_cols].copy()
    y_train = train_df['reoffended_within_180_days'].astype(int)
    X_test = test_df[avail_cols].copy()
    y_test = test_df['reoffended_within_180_days'].astype(int)
    
    numeric_cols = [c for c in avail_cols if c not in ['zone_type', 'weather_conditions']]
    categorical_cols = [c for c in avail_cols if c in ['zone_type', 'weather_conditions']]
    
    for model_name in ['random_forest', 'gradient_boosting', 'logistic_regression']:
        fold_scores = {'roc_auc': [], 'f1': [], 'precision': [], 'recall': []}
        
        cv = StratifiedKFold(n_splits=min(N_SPLITS, len(train_df)//20 + 1), shuffle=True, random_state=RANDOM_STATE)
        
        for train_idx, test_idx in cv.split(X_train, y_train):
            X_tr, X_te = X_train.iloc[train_idx], X_train.iloc[test_idx]
            y_tr, y_te = y_train.iloc[train_idx], y_train.iloc[test_idx]
            
            pipe = build_pipeline(avail_cols, numeric_cols, categorical_cols, model_name)
            pipe.fit(X_tr, y_tr)
            
            y_proba = pipe.predict_proba(X_te)[:, 1]
            y_pred = (y_proba >= 0.5).astype(int)
            
            fold_scores['roc_auc'].append(roc_auc_score(y_te, y_proba))
            fold_scores['f1'].append(f1_score(y_te, y_pred, zero_division=0))
            fold_scores['precision'].append(precision_score(y_te, y_pred, zero_division=0))
            fold_scores['recall'].append(recall_score(y_te, y_pred, zero_division=0))
        
        mean_auc = np.mean(fold_scores['roc_auc'])
        std_auc = np.std(fold_scores['roc_auc'])
        mean_f1 = np.mean(fold_scores['f1'])
        
        ablation_results.append({
            'feature_set': fs_name,
            'model': model_name,
            'roc_auc_mean': round(float(mean_auc), 4),
            'roc_auc_std': round(float(std_auc), 4),
            'f1_mean': round(float(mean_f1), 4),
            'precision_mean': round(float(np.mean(fold_scores['precision'])), 4),
            'recall_mean': round(float(np.mean(fold_scores['recall'])), 4),
        })
        print(f"   {model_name}: ROC-AUC={mean_auc:.4f} (±{std_auc:.4f}), F1={mean_f1:.4f}")

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_json('/tmp/ablation_report.json', orient='records', indent=2)

best_row = ablation_df.loc[ablation_df['roc_auc_mean'].idxmax()]
best_fs = best_row['feature_set']
best_model = best_row['model']

fs_map = {
    'A_original_6': ORIGINAL_6,
    'B_plus_segmentation': ORIGINAL_6 + SEGMENT_FEATURES,
    'C_plus_rules': ORIGINAL_6 + [RULE_FEATURE],
    'D_plus_both': ORIGINAL_6 + SEGMENT_FEATURES + [RULE_FEATURE],
}
best_features = fs_map.get(best_fs, ORIGINAL_6)

print(f"\n✓ Best feature set: {best_fs}")
print(f"✓ Best model: {best_model}")
print(f"✓ Best ROC-AUC: {best_row['roc_auc_mean']:.4f}")

# Train final calibrated model
print(f"\n🢻 Training final calibrated model ({best_model} + {best_fs})...")

X_train = train_df[best_features].copy()
y_train = train_df['reoffended_within_180_days'].astype(int)
X_test = test_df[best_features].copy()
y_test = test_df['reoffended_within_180_days'].astype(int)

numeric_cols = [c for c in best_features if c not in ['zone_type', 'weather_conditions']]
categorical_cols = [c for c in best_features if c in ['zone_type', 'weather_conditions']]

pipe = build_pipeline(best_features, numeric_cols, categorical_cols, best_model)
calibrated = CalibratedClassifierCV(pipe, method='isotonic', cv=3)
calibrated.fit(X_train, y_train)

y_proba = calibrated.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
brier = brier_score_loss(y_test, y_proba)
f1 = f1_score(y_test, y_pred, zero_division=0)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"\nTest ROC-AUC: {auc:.4f}")
print(f"Test PR-AUC: {pr_auc:.4f}")
print(f"Test Brier: {brier:.4f}")
print(f"Test F1: {f1:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")

# Save model as risk_model_v3.joblib (matches ML service expectation)
import joblib
import os
from datetime import datetime

os.makedirs('/tmp/model', exist_ok=True)

model_version = "rf_v3"
model_artifact = {
    'model': calibrated,
    'feature_cols': best_features,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'model_version': model_version,
    'trained_at': datetime.now().isoformat(),
    'data_source': DATA_SOURCE,
    'include_synthetic': INCLUDE_SYNTHETIC,
    'label_horizon_days': LABEL_HORIZON_DAYS,
    'sklearn_version': __import__('sklearn').__version__,
    'metrics': {
        'roc_auc': round(float(auc), 4),
        'pr_auc': round(float(pr_auc), 4),
        'brier': round(float(brier), 4),
        'f1': round(float(f1), 4),
        'precision': round(float(precision), 4),
        'recall': round(float(recall), 4),
    },
    'points_floor_anchors': POINTS_FLOOR_ANCHORS,
    'sanction_thresholds': {
        'warning': RISK_WARNING_THRESHOLD,
        'training': RISK_TRAINING_THRESHOLD,
        'suspension': RISK_SANCTION_THRESHOLD,
    },
}

joblib.dump(model_artifact, '/tmp/model/risk_model_v3.joblib')
print(f"\n✓ Model saved to /tmp/model/risk_model_v3.joblib")
print(f"   Model version: {model_version}")
print(f"   Features: {len(best_features)}")
print(f"   Test ROC-AUC: {auc:.4f}, PR-AUC: {pr_auc:.4f}, Brier: {brier:.4f}")

# 🣊 Step 4c: Classification Metrics

Detailed classification metrics including precision, recall, F1-score, and support per class.
Also includes confusion matrix and macro/micro averaged metrics.

In [ ]:
print("🣊 Classification Metrics Report")
print("=" * 60)

# Classification report with precision, recall, f1-score, support
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Reoffend", "Reoffend"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"  True Negatives  (TN): {cm[0, 0]}")
print(f"  False Positives (FP): {cm[0, 1]}")
print(f"  False Negatives (FN): {cm[1, 0]}")
print(f"  True Positives  (TP): {cm[1, 1]}")

# Macro and micro averages
macro_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
micro_precision = precision_score(y_test, y_pred, average="micro", zero_division=0)
micro_recall = recall_score(y_test, y_pred, average="micro", zero_division=0)
micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)

print("\nMacro-averaged metrics:")
print(f"  Precision: {macro_precision:.4f}")
print(f"  Recall:    {macro_recall:.4f}")
print(f"  F1-Score:  {macro_f1:.4f}")

print("\nMicro-averaged metrics:")
print(f"  Precision: {micro_precision:.4f}")
print(f"  Recall:    {micro_recall:.4f}")
print(f"  F1-Score:  {micro_f1:.4f}")

# Update model_artifact metrics with macro/micro averages
model_artifact["metrics"]["macro_precision"] = round(float(macro_precision), 4)
model_artifact["metrics"]["macro_recall"] = round(float(macro_recall), 4)
model_artifact["metrics"]["macro_f1"] = round(float(macro_f1), 4)
model_artifact["metrics"]["micro_precision"] = round(float(micro_precision), 4)
model_artifact["metrics"]["micro_recall"] = round(float(micro_recall), 4)
model_artifact["metrics"]["micro_f1"] = round(float(micro_f1), 4)

print("\n✓ Classification metrics computed and saved to model artifact.")

# 📝 Step 4b: Heuristic V2 Fallback

Generate heuristic_v2.json for use when the ML service is down.
Contains intercept, coefficients, means, and scales for a deterministic logistic score.

In [ ]:
print("📝 Generating heuristic_v2.json...")

# Compute feature means/scales for heuristic_v2
feature_means = {}
feature_scales = {}
for col in best_features:
    if col in numeric_cols:
        feature_means[col] = round(float(train_df[col].mean()), 4)
        feature_scales[col] = round(float(train_df[col].std()), 4)
    else:
        feature_means[col] = train_df[col].mode().iloc[0] if len(train_df[col].mode()) > 0 else ''
        feature_scales[col] = 1.0

# Non-negative coefficients for heuristic_v2
coefficients = {
    'cumulative_active_demerit_points': 0.015,
    'violation_count_12m': 0.03,
    'violation_count_90d': 0.05,
    'weighted_points_12m': 0.01,
    'severe_offence_share': 0.20,
    'avg_speed_over_limit': 0.005,
    'max_speed_over_limit': 0.003,
    'days_since_last_offense': -0.001,
    'mean_gap_between_offences': -0.001,
    'trend': 0.10,
    'share_night': 0.05,
    'share_school_zone': 0.05,
    'share_bad_weather': 0.05,
    'unpaid_fine_ratio': 0.15,
    'avg_payment_delay_days': 0.002,
    'appeals_lost': 0.05,
    'new_driver': -0.05,
    'speed_recorded': 0.002,
    'past_fine_payment_delays_days': 0.001,
}
intercept = 0.05

heuristic_v2 = {
    'intercept': intercept,
    'coefficients': coefficients,
    'feature_means': feature_means,
    'feature_scales': feature_scales,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'model_version': 'heuristic_v2',
    'description': 'Deterministic logistic fallback when ML service is down. All coefficients non-negative.',
}

heuristic_path = '/tmp/model/heuristic_v2.json'
with open(heuristic_path, 'w') as f:
    json.dump(heuristic_v2, f, indent=2)
print(f"✓ Heuristic v2 saved to {heuristic_path}")

# Save model_meta.json
meta = {
    'model_version': model_version,
    'feature_cols': best_features,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'known_categories': {
        'zone_type': sorted(train_df['zone_type'].dropna().unique().tolist()),
        'weather_conditions': sorted(train_df['weather_conditions'].dropna().unique().tolist()),
    },
    'thresholds': {
        'warning': RISK_WARNING_THRESHOLD,
        'training': RISK_TRAINING_THRESHOLD,
        'suspension': RISK_SANCTION_THRESHOLD,
    },
    'points_floor_anchors': POINTS_FLOOR_ANCHORS,
    'metrics': model_artifact['metrics'],
    'sklearn_version': __import__('sklearn').__version__,
    'trained_at': model_artifact['trained_at'],
    'data_source': DATA_SOURCE,
    'include_synthetic': INCLUDE_SYNTHETIC,
    'label_horizon_days': LABEL_HORIZON_DAYS,
    'n_train': int(len(train_df)),
    'n_test': int(len(test_df)),
}

meta_path = '/tmp/model/model_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"✓ Model metadata saved to {meta_path}")

# ⚖️ Step 5: Fairness Report

Compute false-positive rate (FPR) by zone_type, weather_conditions, and segment.
Flag any group whose FPR exceeds 1.25× the overall rate.

In [ ]:
print("⚖️ Fairness Report")
print("=" * 60)

X_train_full = train_df[best_features].copy()
y_train_full = train_df['reoffended_within_180_days'].astype(int)
y_train_proba = calibrated.predict_proba(X_train_full)[:, 1]
y_train_pred = (y_train_proba >= 0.5).astype(int)

train_df['y_pred'] = y_train_pred
train_df['y_true'] = y_train_full

overall_fpr = ((train_df['y_pred'] == 1) & (train_df['y_true'] == 0)).sum() / max((train_df['y_true'] == 0).sum(), 1)
print(f"Overall FPR: {overall_fpr:.4f}")

fairness = {'overall_fpr': round(float(overall_fpr), 4)}

for group_col in ['zone_type', 'weather_conditions']:
    if group_col not in train_df.columns:
        continue
    group_fairness = {}
    for val in train_df[group_col].unique():
        subset = train_df[train_df[group_col] == val]
        if len(subset) == 0:
            continue
        fpr = ((subset['y_pred'] == 1) & (subset['y_true'] == 0)).sum() / max((subset['y_true'] == 0).sum(), 1)
        tpr = ((subset['y_pred'] == 1) & (subset['y_true'] == 1)).sum() / max((subset['y_true'] == 1).sum(), 1)
        flagged = fpr > overall_fpr * 1.25
        group_fairness[str(val)] = {
            'fpr': round(float(fpr), 4),
            'tpr': round(float(tpr), 4),
            'n': int(len(subset)),
            'fpr_ratio_to_overall': round(float(fpr / max(overall_fpr, 0.001)), 4),
            'flagged': bool(flagged),
        }
    fairness[group_col] = group_fairness
    flagged_groups = [k for k, v in group_fairness.items() if v['flagged']]
    if flagged_groups:
        print(f"  {group_col}: FLAGGED groups: {flagged_groups}")
    else:
        print(f"  {group_col}: No groups flagged")

import json as json_lib
with open('/tmp/model/fairness_report.json', 'w') as f:
    json_lib.dump(fairness, f, indent=2)

print(f"\n✓ Fairness report saved to /tmp/model/fairness_report.json")

# 📥 Step 6: Download Artifacts

Download the trained model and reports. Copy them to your Laravel `ml_service/model/` and `ml_service/reports/` directories.

Artifacts included:
- `risk_model_v3.joblib` → `ml_service/model/risk_model_v3.joblib`
- `model_meta.json` → `ml_service/model/model_meta.json`
- `heuristic_v2.json` → `ml_service/model/heuristic_v2.json`
- `ablation_report.json` → `ml_service/reports/ablation_report.json`
- `fairness_report.json` → `ml_service/reports/fairness_report.json`

In [ ]:
import zipfile
from pathlib import Path
import shutil

output_dir = Path('/tmp/model_output')
output_dir.mkdir(parents=True, exist_ok=True)

# Model
model_src = Path('/tmp/model/risk_model_v3.joblib')
if model_src.exists():
    shutil.copy2(model_src, output_dir / 'risk_model_v3.joblib')
    print(f"✓ Copied risk_model_v3.joblib")
else:
    print(f"⚠️  Model file not found at {model_src}")

# Metadata
meta_src = Path('/tmp/model/model_meta.json')
if meta_src.exists():
    shutil.copy2(meta_src, output_dir / 'model_meta.json')
    print(f"✓ Copied model_meta.json")
else:
    print(f"⚠️  Model metadata not found at {meta_src}")

# Heuristic v2
heuristic_src = Path('/tmp/model/heuristic_v2.json')
if heuristic_src.exists():
    shutil.copy2(heuristic_src, output_dir / 'heuristic_v2.json')
    print(f"✓ Copied heuristic_v2.json")
else:
    print(f"⚠️  Heuristic v2 not found at {heuristic_src}")

# Reports
for report_file in ['/tmp/model/fairness_report.json', '/tmp/ablation_report.json']:
    src = Path(report_file)
    if src.exists():
        shutil.copy2(src, output_dir / src.name)
        print(f"✓ Copied {src.name}")
    else:
        print(f"⚠️  {report_file} not found")

# Create zip
zip_path = Path('/tmp/risk_model_artifacts.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in output_dir.rglob('*'):
        if p.is_file():
            zf.write(p, p.name)

print(f"\n✓ Artifacts zipped to: {zip_path}")
print(f"   Size: {zip_path.stat().st_size / 1024:.1f} KB")

from google.colab import files
files.download(str(zip_path))

print("\n📋 Next steps:")
print("1. Download the zip above")
print("2. Extract risk_model_v3.joblib to your Laravel ml_service/model/ directory")
print("3. Extract model_meta.json and heuristic_v2.json to ml_service/model/")
print("4. Extract ablation_report.json and fairness_report.json to ml_service/reports/")
print("5. Restart the FastAPI service: uvicorn app:app --host 127.0.0.1 --port 8001 --reload")

# 📋 Demo Checklist

## Before running the notebook:

1. **Export three CSVs from phpMyAdmin (or mysql CLI):**
   - `violations.csv` — from the `violations` table
   - `payments.csv` — from the `payments` table
   - `drivers.csv` — from the `driver_profiles` table

2. **Open this notebook in Google Colab**

3. **Run all cells** (Runtime → Run all)

4. **Download the artifacts zip** at the end

5. **Deploy to Laravel:**
   - Copy `risk_model_v3.joblib` to `apps/backend/traffic-demerit-web/ml_service/model/`
   - Copy `model_meta.json` and `heuristic_v2.json` to `apps/backend/traffic-demerit-web/ml_service/model/`
   - Copy `ablation_report.json` and `fairness_report.json` to `apps/backend/traffic-demerit-web/ml_service/reports/`
   - Restart the ML service: `python -m uvicorn app:app --host 127.0.0.1 --port 8001 --reload`

## Edge cases handled:
- Drivers with no violations (default 365 days since last offense)
- Missing payments.csv → payment features default to 0
- Missing/odd column names → auto-mapped via aliases
- Mixed-case zone/weather values → matched as-is
- Unparseable dates → dropped with warning
- Fewer than 200 real violations → warning, set INCLUDE_SYNTHETIC = True

## Data provenance:
Results are based on your exported database. If `INCLUDE_SYNTHETIC = True`, seeded/demo rows are included and the model may learn seeder patterns rather than real-world behavior.